# Perhitungan DS Awal dan DS–PSO

Notebook ini menghitung ulang metrik dari data uji yang sama untuk DS awal dan 10 run PSO (seed 42–51). Fungsi inferensi, metrik, dan optimasi memakai `core/run_experiment.py`, yaitu program yang menghasilkan laporan eksperimen. Dataset dipatok ke ekspor yang tercatat dalam `data/experiment_report.json`; aturan awal dipatok ke `data/belief.json` untuk optimasi. Jalankan semua sel dari atas ke bawah; optimasi 10 × 30 partikel × 100 iterasi memerlukan waktu.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from core.run_experiment import ExperimentEngine, constrain, file_hash, metrics, optimize

report_path = Path("data/experiment_report.json")
arsip = json.loads(report_path.read_text(encoding="utf-8"))
metadata = arsip["metadata"]
train_path = Path(metadata["train"])
test_path = Path(metadata["test"])
belief_path = Path("data/belief.json")
for path in (train_path, test_path, belief_path):
    if not path.is_file():
        raise FileNotFoundError(path)
if file_hash(train_path) != metadata["train_sha256"] or file_hash(test_path) != metadata["test_sha256"]:
    raise ValueError("Dataset tidak cocok dengan metadata laporan.")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
rules = json.loads(belief_path.read_text(encoding="utf-8"))["rules"]
classes = sorted(df_train["Kerusakan"].unique().tolist())
if set(df_test["Kerusakan"]) - set(classes):
    raise ValueError("Ada kelas uji yang tidak terdapat pada data latih.")
counts = df_train["Kerusakan"].value_counts()
priors = {name: float(counts.get(name, 0) / len(df_train)) for name in classes}
engine = ExperimentEngine(rules, classes, priors)
train_active = engine.active_rules(df_train)
test_active = engine.active_rules(df_test)
initial = constrain([hyp["initial_belief"] for rule in rules for hyp in rule["hypotheses"]], rules)

display(pd.DataFrame({"Berkas": [str(train_path), str(test_path)], "Jumlah kasus": [len(df_train), len(df_test)]}))
print(f"Aturan: {len(rules)}; parameter belief: {len(initial)}; kelas: {len(classes)}")

## 1. DS awal pada data uji

Untuk tiap kasus, aturan aktif digabung dengan belief pakar lalu massa Θ dibagi ke seluruh kelas (probabilitas pignistik). Prediksi ialah kelas dengan probabilitas tertinggi. Accuracy = benar / jumlah uji; F1 macro merata-ratakan F1 setiap kelas; F1 weighted memakai support kelas. Multiclass Brier = rata-rata jumlah kuadrat `(probabilitas kelas - one-hot label aktual)^2` per kasus.

In [ ]:
y_test = df_test["Kerusakan"].to_numpy()
baseline_prob = engine.matrix(test_active, initial)
baseline = metrics(y_test, baseline_prob, classes)
benar_ds = round(baseline["accuracy"] * len(y_test))
fmt_persen = lambda value: f"{value:.2%}".replace(".", ",")
print(f"Accuracy DS awal = {benar_ds}/{len(y_test)} x 100% = {fmt_persen(baseline['accuracy'])}")
display(pd.DataFrame({
    "Metrik": ["Accuracy", "Macro F1", "Weighted F1", "Brier score"],
    "DS awal": [fmt_persen(baseline["accuracy"]), fmt_persen(baseline["macro_f1"]),
                fmt_persen(baseline["weighted_f1"]), f"{baseline['multiclass_brier']:.4f}".replace(".", ",")],
}))

## 2. Optimasi PSO dan evaluasi setiap seed

Fitness tiap partikel dihitung **hanya pada data latih** sebagai `1 - multiclass Brier`. Setiap seed menjalankan 30 partikel dan 100 iterasi. Setelah optimasi, belief terbaik seed tersebut dievaluasi pada data uji dengan fungsi metrik yang sama seperti DS awal.

In [ ]:
seeds = list(range(42, 52))
particles, iterations = 30, 100
runs = []
for seed in seeds:
    beliefs, fitness, history = optimize(
        engine, train_active, df_train["Kerusakan"].to_numpy(), rules,
        initial, seed, particles, iterations,
    )
    result = metrics(y_test, engine.matrix(test_active, beliefs), classes)
    runs.append({"seed": seed, "beliefs": beliefs, "train_fitness": fitness,
                 "history": history, "metrics": result})
    print(f"Seed {seed}: fitness latih={fitness:.6f}, accuracy uji={result['accuracy']:.4%}, "
          f"macro F1 uji={result['macro_f1']:.4%}")

per_seed = pd.DataFrame([{
    "Seed": run["seed"], "Fitness latih": run["train_fitness"],
    "Accuracy": run["metrics"]["accuracy"], "Macro F1": run["metrics"]["macro_f1"],
    "Weighted F1": run["metrics"]["weighted_f1"],
    "Brier score": run["metrics"]["multiclass_brier"],
} for run in runs])
display(per_seed.style.format({"Fitness latih": "{:.6f}", "Accuracy": "{:.2%}",
                               "Macro F1": "{:.2%}", "Weighted F1": "{:.2%}",
                               "Brier score": "{:.4f}"}))

## 3. Rata-rata, seed terbaik

In [ ]:
best = max(runs, key=lambda run: (run["metrics"]["macro_f1"], run["metrics"]["accuracy"]))
summary = per_seed[["Accuracy", "Macro F1", "Weighted F1", "Brier score"]].agg(["mean", "std"])
display(summary.style.format("{:.6f}"))
print(f"Seed terbaik: {best['seed']}; prediksi benar: "
      f"{round(best['metrics']['accuracy'] * len(y_test))}/{len(y_test)}")

mapping = [("Accuracy", "accuracy"), ("Macro F1", "macro_f1"),
           ("Weighted F1", "weighted_f1"), ("Brier score", "multiclass_brier")]
comparison = pd.DataFrame([{
    "Metrik": label,
    "DS awal": baseline[key],
    "DS PSO rata-rata": summary.loc["mean", label],
    "DS PSO terbaik": best["metrics"][key],
    "Perubahan terbaik dari DS": (baseline[key] - best["metrics"][key] if key == "multiclass_brier"
                                  else round(best["metrics"][key] * 100, 2) - round(baseline[key] * 100, 2)),
} for label, key in mapping]).set_index("Metrik")
comparison_display = comparison.astype(object).copy()
for label in comparison_display.index:
    for column in comparison_display.columns:
        value = comparison_display.loc[label, column]
        if label == "Brier score":
            comparison_display.loc[label, column] = f"{value:.4f}"
        elif column == "Perubahan terbaik dari DS":
            comparison_display.loc[label, column] = f"{value:+.2f} poin persentase"
        else:
            comparison_display.loc[label, column] = f"{value:.2%}"
display(comparison_display)
print("Perubahan Brier = Brier DS awal - Brier PSO terbaik (turun).")